# Notebook 4: LLM-as-a-Judge (Zero-Shot, Few Shot, Persona CoT)
## Project: Multi-Paradigm AI vs. Human Text Detection

### Objective
To test whether modern LLMs can act as their own detectors without fine-tuning. We evaluate smaller, efficient instruction-tuned models (like Phi-2, Qwen) using three distinct prompting strategies:
1. **Zero-Shot:** Asking the model directly for a classification.
2. **Persona Chain-of-Thought (CoT):** Forcing the model to adopt the persona of a forensic linguist and reason step-by-step before outputting a decision.
3. **Few-Shot:** Sharing examples with LLMs to understand context and tehn classify texts.

In [107]:
!pip install -q transformers accelerate bitsandbytes

In [108]:
# Importing libraries
import torch
import pandas as pd
from transformers import AutoConfig,AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
from tqdm.notebook import tqdm
import re
import random
import numpy as np
import gc
import warnings
warnings.filterwarnings("ignore")

In [109]:
# Maintaining reusability
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

In [110]:
# Loading data
df=pd.read_csv("/kaggle/input/datasets/matejkore/ai-detector-dataset/ai_detector_dataset.csv")
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",Human
1,Love this bar! Fun crowd and the staff are all...,Human
2,"After watching Whale Wars: Viking Shores, I ca...",Human
3,Kelly really wanted the new iPhone. She begged...,Human
4,Though this is the easiest way to secure a spo...,Human


In [111]:
# Replacing author names with binary labels
df['label']=[1 if author=='AI' else 0 for author in df['label']]
df.head()

,text,label
0,"Got take-out. Very friendly staff, reasonable ...",0
1,Love this bar! Fun crowd and the staff are all...,0
2,"After watching Whale Wars: Viking Shores, I ca...",0
3,Kelly really wanted the new iPhone. She begged...,0
4,Though this is the easiest way to secure a spo...,0


In [112]:
# Dropping duplicates
df=df.drop_duplicates(subset='text')
df.duplicated(subset='text').sum()

np.int64(0)

In [113]:
# Sampling data from the dataset
subset_df=df.sample(50,random_state=42)
subset_df.shape

(50, 2)

In [114]:
# Setting up quantisation
bnb_config=BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.float16,
                              bnb_4bit_use_double_quant=True)

In [115]:
strategies={
    "Zero-Shot": "Instruct: Classify the following text as either 'AI' or 'Human'.\nText: {text}\nFinal Answer: [[",
    "Persona_CoT": """Instruct: You are a linguistics expert. Analyze the style, vocabulary, and flow of the text below. 
Determine if it is AI-generated or Human-written. Think step-by-step, then provide your final answer in the format: Final Answer: [[Label]]
Text: {text}
Analysis:""",
    "Few-Shot": """Instruct: Classify the text as 'AI' or 'Human'.
Example 1: "Got take-out. Very friendly staff, reasonable prices, good food. I will be back! As I waited for my order, I saw the huge portions they served to dine in guests. It was clean and smelled delicious." -> [[Human]]
Example 2: "Roy had a very dirty bathroom. So he bought some industrial grade cleaners to clean it. His mother warned him about the dangers of such chemicals. He used the chemicals for all purpose and cleaned the bathroom." -> [[AI]]
Example 3: "A man put on a button. It supported a candidate. A stranger saw it. The stranger commented on the button. The man nodded in approval." -> [[Human]]
Example 4: "To act like her, be friendly and outgoing, and introduce yourself to lots of new people. [substeps] If there is anyone in your school classes you don't know, for example, Make sure you are nice and silly." -> [[AI]]
Text: {text}
Final Answer: [["""
}

In [116]:
# Prediction and parsing function
def get_research_prediction(text,strategy_name,model,tokenizer):
    cleaned_text=" ".join(text.split()[:250]) 
    prompt=strategies[strategy_name].format(text=cleaned_text)
    inputs=tokenizer(prompt,return_tensors="pt",return_attention_mask=True).to("cuda")
    with torch.no_grad():
        max_tokens=100 if strategy_name=="Persona_CoT" else 5
        outputs=model.generate(**inputs, 
                               max_new_tokens=max_tokens,
                               pad_token_id=tokenizer.eos_token_id,
                               temperature=0.1,
                               do_sample=True)
    response=tokenizer.decode(outputs[0],skip_special_tokens=True)
    if "[[" in response:
        result=response.split("[[")[-1].split("]]")[0].lower()
        if 'human' in result: 
            return 0
        if 'ai' in result: 
            return 1
    last_bit=response[-15:].lower()
    if 'human' in last_bit: 
        return 0
    if 'ai' in last_bit: 
        return 1
    return 0

In [117]:
# Executing Multi-Model Multi-Strategy Benchmark
model_list=["microsoft/phi-2", "Qwen/Qwen2-1.5B-Instruct"]
for model_id in model_list:
    print(f"\n{'='*40}\nMODEL: {model_id}\n{'='*40}")
    tokenizer=AutoTokenizer.from_pretrained(model_id,trust_remote_code=True)
    tokenizer.pad_token=tokenizer.eos_token
    config=AutoConfig.from_pretrained(model_id,trust_remote_code=True)
    config.pad_token_id=tokenizer.eos_token_id
    model=AutoModelForCausalLM.from_pretrained(model_id, 
                                               config=config, 
                                               quantization_config=bnb_config, 
                                               device_map="auto", 
                                               trust_remote_code=True)
    for strat_name in strategies.keys():
        col_name=f"{model_id.split('/')[-1]}_{strat_name}"
        print(f"Running Strategy: {strat_name}...")
        subset_df[col_name]=subset_df['text'].progress_apply(lambda x: get_research_prediction(x,strat_name,model,tokenizer))
        acc=(subset_df['label']==subset_df[col_name]).mean()
        print(f"-> Accuracy: {acc*100:.2f}%")
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()


MODEL: microsoft/phi-2


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Running Strategy: Zero-Shot...


  0%|          | 0/50 [00:00<?, ?it/s]

-> Accuracy: 42.00%
Running Strategy: Persona_CoT...


...

-> Accuracy: 38.00%
Running Strategy: Persona_CoT...


  0%|          | 0/50 [00:00<?, ?it/s]

-> Accuracy: 52.00%
Running Strategy: Few-Shot...


  0%|          | 0/50 [00:00<?, ?it/s]

-> Accuracy: 40.00%
